# COMP 469 — Lab 01: Vacuum World Agents
**CSUCI · Introduction to Artificial Intelligence · AIMA Chapters 1–2**

---

## Lab Goal

You will build a Vacuum World environment from scratch and then program three increasingly capable agents:

| Agent type | What it knows | How it decides |
|---|---|---|
| **Simple reflex** | Only the current percept | Condition-action rules, no memory |
| **Model-based** | Current percept + internal memory | Tracks visited squares, avoids backtracking |
| **Goal-based** | Current percept + a desired future state | Plans a path toward dirty squares using BFS |

By the end, you should be able to explain—using AIMA vocabulary—how agent design depends on **percepts**, **actions**, the **performance measure**, and **environment properties**.

---

## What You Will Submit

A completed notebook that:

- Runs **top to bottom without errors** after a fresh *Kernel → Restart & Run All*
- Has completed PEAS and environment-property tables (Section 2)
- Has working environment code (Section 3)
- Has working reflex, model-based, and goal-based agents (Sections 7–9)
- Shows experiment results and at least two plots (Sections 10–11)
- Has written answers to **all five checkpoints** and a final reflection

---

## Before You Start

> **Kernel check:** In the Jupyter menu, confirm the kernel shown in the top-right corner reads **COMP 469 (.env)**.  
> If it says something else (e.g., *Python 3*), click it and switch to the correct kernel.  
> If the kernel is missing, re-run the `.env` setup below.

---

## Python `.env` Setup (First Time Only)

Run these commands **in a terminal** from the folder where this notebook is located.

**macOS / Linux**
```bash
python3 -m venv .env
source .env/bin/activate
python -m pip install --upgrade pip
python -m pip install ipykernel notebook jupyter matplotlib
python -m ipykernel install --user --name comp469 --display-name "COMP 469 (.env)"
```

**Windows PowerShell**
```powershell
py -m venv .env
# If the next line is blocked, first run:
#   Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process
.\.env\Scripts\Activate.ps1
python -m pip install --upgrade pip
python -m pip install ipykernel notebook jupyter matplotlib
python -m ipykernel install --user --name comp469 --display-name "COMP 469 (.env)"
```

**Every subsequent session**, activate before launching Jupyter:
```bash
source .env/bin/activate   # macOS/Linux
jupyter notebook
```
```powershell
.\.env\Scripts\Activate.ps1   # Windows
jupyter notebook
```


---
## 1. Setup Cell

Run this cell first. Do not edit it.


In [ ]:
import random
from collections import deque
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt

Position = Tuple[int, int]
ACTIONS = ["Suck", "Up", "Down", "Left", "Right", "NoOp"]

random.seed(469)
print("Setup complete. You are ready for Lab 01.")


---
## 2. PEAS Analysis

Before writing any code, analyze the task environment using the PEAS framework (AIMA §2.3).

Complete both tables. Use precise AIMA vocabulary — avoid vague answers like "the grid."

### 2a. PEAS Table

| PEAS Component | Your Answer |
|---|---|
| **Performance measure** | *TODO — what numeric criteria should a rational agent maximize?* |
| **Environment** | *TODO — what is the physical setting the agent operates in?* |
| **Actuators** | *TODO — what outputs/actions can the agent produce?* |
| **Sensors** | *TODO — what can the agent perceive at each step?* |

---

### 2b. Environment Properties

Classify this environment using AIMA vocabulary (§2.3.2). Give a one-sentence justification for each choice — "yes/no" answers without reasoning earn no credit.

| Property | Your Choice | Brief justification (1 sentence) |
|---|---|---|
| Fully / Partially observable? | *TODO* | *TODO* |
| Single-agent / Multi-agent? | *TODO* | *TODO* |
| Deterministic / Stochastic? | *TODO* | *TODO* |
| Episodic / Sequential? | *TODO* | *TODO* |
| Static / Dynamic? | *TODO* | *TODO* |
| Discrete / Continuous? | *TODO* | *TODO* |
| Known / Unknown? | *TODO* | *TODO* |

---

### ✏️ Checkpoint 1

> In 3–5 sentences, explain why the **same physical grid** could lead to different agent designs if the sensors or performance measure changed. Use AIMA vocabulary and give at least one concrete example from this lab.

*Your answer here.*


---
## 3. Build the Vacuum World Environment

The environment is the **ground truth** of the world. The agent never sees it directly — it only receives a `Percept`.

### Scoring rules (implement these exactly)
| Event | Score change |
|---|---|
| Any action taken | −1 |
| Cleaning a dirty square (`Suck` on a `"Dirty"` square) | +10 |
| Moving into a wall or obstacle (bump) | −2 additional |
| `Suck` on a clean square | just the −1 for the action |

### Coordinate system
```
(0,0) ── x increases right ──▶
  │
  y increases downward
  ▼
```
So `"Up"` means `y − 1`, and `"Down"` means `y + 1`.

### What you need to complete

There are **four TODOs** in the class below (labeled TODO 1 – TODO 4). Read each docstring carefully before writing code.

After completing them, the smoke test cell should print a nonzero dirty-square count and obstacle count.


In [ ]:
@dataclass
class Percept:
    """The only information the agent receives each step."""
    location: Position   # agent's current (x, y)
    status: str          # "Clean", "Dirty"


class VacuumEnvironment:
    def __init__(
        self,
        width: int = 5,
        height: int = 5,
        dirt_probability: float = 0.30,
        obstacle_probability: float = 0.0,
        start: Position = (0, 0),
        seed: Optional[int] = None,
    ):
        if seed is not None:
            random.seed(seed)

        self.width = width
        self.height = height
        self.start = start
        self.agent_location = start
        self.status: Dict[Position, str] = {}
        self.obstacles: set = set()
        self.score = 0
        self.steps = 0
        self.cleaned_count = 0

        # ── TODO 1: Place obstacles ────────────────────────────────────
        # Iterate over every grid location.
        # For each location (that is NOT the start), add it to
        # self.obstacles with probability obstacle_probability.
        # Use: random.random() < obstacle_probability
        #
        # Constraint: never place an obstacle at self.start.
        #
        # Hint: obstacles will be used in TODO 2 and TODO 4.
        for x in range(width):
            for y in range(height):
                location = (x, y)
                # ── your code here ──
                pass
        # ──────────────────────────────────────────────────────────────

        # ── TODO 2: Initialize square status ──────────────────────────
        # Iterate over every grid location.
        # If the location is an obstacle: self.status[location] = "Obstacle"
        # Otherwise:  assign "Dirty" with probability dirt_probability,
        #             "Clean" otherwise.
        # Use: random.random() < dirt_probability
        #
        # After the loop, force self.status[start] = "Clean"
        # (the agent always starts on a clean square).
        for x in range(width):
            for y in range(height):
                location = (x, y)
                # ── your code here ──
                self.status[location] = "Clean"   # ← replace this stub
        self.status[start] = "Clean"
        # ──────────────────────────────────────────────────────────────

    # ── Helper methods (do not edit) ──────────────────────────────────
    def in_bounds(self, location: Position) -> bool:
        x, y = location
        return 0 <= x < self.width and 0 <= y < self.height

    def is_accessible(self, location: Position) -> bool:
        """True if location is inside the grid AND not an obstacle."""
        return self.in_bounds(location) and location not in self.obstacles

    def get_percept(self) -> Percept:
        """Return the agent's current percept (location + square status)."""
        return Percept(self.agent_location, self.status[self.agent_location])

    def get_neighbors(self, location: Position) -> Dict[str, Position]:
        """Return only the accessible neighbors as {action: next_location}."""
        x, y = location
        candidates = {
            "Up":    (x, y - 1),
            "Down":  (x, y + 1),
            "Left":  (x - 1, y),
            "Right": (x + 1, y),
        }
        return {
            action: next_loc
            for action, next_loc in candidates.items()
            if self.is_accessible(next_loc)
        }

    def execute_action(self, action: str) -> None:
        """Apply one action to the environment and update the score."""
        if action not in ACTIONS:
            raise ValueError(f"Unknown action: {action}")

        self.steps += 1
        self.score -= 1   # every action costs 1 point

        if action == "NoOp":
            return

        # ── TODO 3: Implement Suck ────────────────────────────────────
        # If action == "Suck":
        #   - If the current square is "Dirty":
        #       set self.status[self.agent_location] = "Clean"
        #       add 10 to self.score
        #       increment self.cleaned_count
        #   Return after handling Suck (do NOT fall through to movement).
        #
        # ── your code here ──


        # ──────────────────────────────────────────────────────────────

        # ── TODO 4: Implement movement ────────────────────────────────
        # Build a moves dict: {"Up": (x,y-1), "Down": (x,y+1), ...}
        # Look up next_location = moves[action].
        # If next_location is accessible (use self.is_accessible):
        #   update self.agent_location
        # Else (wall or obstacle bump):
        #   subtract 2 more from self.score
        #
        # Coordinate reminder: "Up" decreases y, "Down" increases y.
        #
        # ── your code here ──


        # ──────────────────────────────────────────────────────────────

    # ── Utility methods (do not edit) ─────────────────────────────────
    def count_dirty_squares(self) -> int:
        return sum(1 for v in self.status.values() if v == "Dirty")

    def count_clean_squares(self) -> int:
        return sum(
            1 for loc, v in self.status.items()
            if v == "Clean" and loc not in self.obstacles
        )

    def is_done(self) -> bool:
        return self.count_dirty_squares() == 0

    def copy_world(self):
        """Snapshot the current grid state for reuse across agents."""
        return dict(self.status), set(self.obstacles)

    def load_world(self, status: Dict[Position, str], obstacles: set) -> None:
        """Restore a previously snapshotted grid state."""
        self.status = dict(status)
        self.obstacles = set(obstacles)
        self.agent_location = self.start
        self.score = 0
        self.steps = 0
        self.cleaned_count = 0


### Smoke test — run after completing TODOs 1–4

In [ ]:
env = VacuumEnvironment(
    width=5, height=5,
    dirt_probability=0.35,
    obstacle_probability=0.10,
    seed=469,
)
print("Start location:", env.agent_location)
print("Percept at start:", env.get_percept())
print("Dirty squares:", env.count_dirty_squares())   # should be > 0
print("Obstacles placed:", len(env.obstacles))        # should be > 0

# Expected with seed=469:
#   Dirty squares: 8
#   Obstacles:     3


---
## 4. Visualize and Inspect the World

These functions are pre-written for your debugging convenience. Run this cell and confirm the grid looks reasonable before moving on.

**Grid legend**
| Symbol / Color | Meaning |
|---|---|
| `A` / blue dot | Agent's current location |
| `D` / dark gray | Dirty square |
| `.` / light gray | Clean square |
| `#` / black | Obstacle |


In [ ]:
def display_text(env: VacuumEnvironment) -> None:
    """Print the grid as ASCII art (row 0 at top)."""
    for y in range(env.height):
        row = []
        for x in range(env.width):
            loc = (x, y)
            if loc == env.agent_location:
                row.append("A")
            elif loc in env.obstacles:
                row.append("#")
            elif env.status[loc] == "Dirty":
                row.append("D")
            else:
                row.append(".")
        print(" ".join(row))
    print(f"Score: {env.score}  Steps: {env.steps}  Dirty left: {env.count_dirty_squares()}")


def plot_environment(env: VacuumEnvironment, title: str = "Vacuum World") -> None:
    """Render a color-coded grid using matplotlib."""
    grid = []
    for y in range(env.height):
        row = []
        for x in range(env.width):
            loc = (x, y)
            if loc in env.obstacles:
                row.append(2)
            elif env.status[loc] == "Dirty":
                row.append(1)
            else:
                row.append(0)
        grid.append(row)

    plt.figure(figsize=(5, 5))
    plt.imshow(grid, cmap="Greys", vmin=0, vmax=2)
    ax = plt.gca()
    ax.set_xticks(range(env.width))
    ax.set_yticks(range(env.height))
    ax.set_xticks([i - 0.5 for i in range(1, env.width)], minor=True)
    ax.set_yticks([i - 0.5 for i in range(1, env.height)], minor=True)
    ax.grid(which="minor", color="black", linewidth=1)
    ax.scatter([env.agent_location[0]], [env.agent_location[1]], c="tab:blue", s=250, zorder=5)
    ax.set_title(title)
    plt.show()


display_text(env)
plot_environment(env, "Initial Vacuum World (seed=469)")


---
## 5. Agent Interface and Random Baseline

Every agent implements a single method: `choose_action(percept, environment) -> str`.

The **random agent** is provided as a baseline and a sanity-check. It:
- Returns `"Suck"` if the current square is dirty
- Otherwise picks a random cardinal direction

Notice it receives the full `environment` object — but it only uses `percept`. Later agents may legitimately use more of the environment object (particularly the goal-based agent in Section 9).


In [ ]:
class Agent:
    """Abstract base class. All agents must implement choose_action."""
    name = "Base Agent"

    def choose_action(self, percept: Percept, environment: VacuumEnvironment) -> str:
        raise NotImplementedError


class RandomVacuumAgent(Agent):
    """
    A simple-reflex baseline. Cleans when dirty; moves randomly otherwise.
    This agent has NO internal state — it is stateless between calls.
    """
    name = "Random Vacuum Agent"

    def choose_action(self, percept: Percept, environment: VacuumEnvironment) -> str:
        if percept.status == "Dirty":
            return "Suck"
        return random.choice(["Up", "Down", "Left", "Right"])


# Quick sanity check
random_agent = RandomVacuumAgent()
print(random_agent.name, "→", random_agent.choose_action(env.get_percept(), env))


---
## 6. Simulation Runner

The runner repeatedly calls the agent and steps the environment until the world is clean or `max_steps` is reached.

**This cell is pre-written.** Read through it carefully — you'll need to understand its return dictionary when you analyze results in Section 10.

The returned `dict` contains:
- `"score"` — final score
- `"steps"` — total steps taken
- `"cleaned"` — number of squares successfully cleaned
- `"dirty_left"` — dirty squares remaining at end
- `"score_history"` — list of cumulative scores per step (useful for plotting)
- `"dirty_history"` — list of dirty-square counts per step


In [ ]:
def run_simulation(
    agent: Agent,
    env: VacuumEnvironment,
    max_steps: int = 100,
    verbose: bool = False,
) -> dict:
    score_history = []
    dirty_history = []
    action_history = []

    for step in range(max_steps):
        percept = env.get_percept()
        action = agent.choose_action(percept, env)

        if action == "NoOp":
            break

        env.execute_action(action)

        score_history.append(env.score)
        dirty_history.append(env.count_dirty_squares())
        action_history.append(action)

        if verbose:
            print(
                f"step={step:02d}  loc={percept.location}  "
                f"status={percept.status:<6}  action={action:<6}  score={env.score}"
            )

        if env.is_done():
            break

    return {
        "agent":          agent.name,
        "score":          env.score,
        "steps":          env.steps,
        "cleaned":        env.cleaned_count,
        "dirty_left":     env.count_dirty_squares(),
        "score_history":  score_history,
        "dirty_history":  dirty_history,
        "action_history": action_history,
    }


# Test run (verbose so you can follow what's happening)
test_env = VacuumEnvironment(
    width=5, height=5, dirt_probability=0.35, obstacle_probability=0.0, seed=101
)
test_result = run_simulation(RandomVacuumAgent(), test_env, max_steps=25, verbose=True)
print("\nSummary:", {k: v for k, v in test_result.items() if k not in ("score_history","dirty_history","action_history")})


---
## 7. Simple Reflex Agent

### Concept

A **simple reflex agent** (AIMA §2.4.1) maps percepts directly to actions via condition-action rules:

```
if <condition on current percept>:  return <action>
```

It has **no internal state** — no memory of previous percepts, no history, no model of the world. Every call to `choose_action` is independent.

> **Design constraint:** Your `ReflexVacuumAgent` must have **no instance variables** (`self.anything`) other than what the base class provides. Storing `self.visited`, `self.steps`, or anything similar disqualifies it as a reflex agent.

### Your task

Complete `choose_action` so that:

1. If the current square is dirty → return `"Suck"` ✓ *(already done)*
2. Otherwise → return a **deterministic** movement action computed **solely from `percept.location`**

**Suggested strategy — serpentine sweep:**

```
row 0: → → → → ↓
row 1: ← ← ← ← ↓
row 2: → → → → ↓
...
```

On even rows (`y % 2 == 0`), the agent should prefer moving `"Right"`.  
At the right edge (`x == width − 1`), prefer `"Down"`.  
On odd rows, prefer `"Left"`. At the left edge (`x == 0`), prefer `"Down"`.

You may also design your own deterministic rule — just document it clearly in Checkpoint 2.

> **Obstacle handling:** `environment.get_neighbors(percept.location)` returns only *accessible* neighbors. Use it to check whether your preferred direction is legal before returning it, and fall back to another legal action if blocked.


In [ ]:
class ReflexVacuumAgent(Agent):
    """
    Simple reflex agent: maps (location, status) → action.
    NO instance variables allowed (other than those inherited from Agent).
    """
    name = "Reflex Vacuum Agent"

    def choose_action(self, percept: Percept, environment: VacuumEnvironment) -> str:
        x, y = percept.location

        # Rule 1: always clean if dirty
        if percept.status == "Dirty":
            return "Suck"

        # ── TODO 6: Implement a deterministic movement rule ────────────
        #
        # Requirements:
        #   - Must be deterministic (same percept → same action every time)
        #   - Must use ONLY percept.location (no stored state)
        #   - Must handle obstacles gracefully using get_neighbors()
        #
        # Serpentine template (fill in the blanks):
        #
        #   legal = environment.get_neighbors(percept.location)
        #   if y % 2 == 0:       # even row: sweep right
        #       preferred = ???
        #   else:                 # odd row: sweep left
        #       preferred = ???
        #
        #   if preferred in legal:
        #       return preferred
        #
        #   # Obstacle fallback: try directions in a fixed priority order
        #   for action in ["Right", "Down", "Left", "Up"]:
        #       if action in legal:
        #           return action
        #
        #   return "NoOp"
        #
        # ── your code here ──
        pass   # ← remove this once you've written your rule
        # ──────────────────────────────────────────────────────────────


# Test your reflex agent on a clean grid (no obstacles to interfere)
reflex_env = VacuumEnvironment(
    width=5, height=5, dirt_probability=0.35, obstacle_probability=0.0, seed=202
)
reflex_result = run_simulation(ReflexVacuumAgent(), reflex_env, max_steps=100, verbose=False)
print("Reflex agent result:")
print({k: v for k, v in reflex_result.items() if k not in ("score_history","dirty_history","action_history")})


### ✏️ Checkpoint 2

Answer in **4–6 sentences** using AIMA vocabulary:

1. What information does your reflex agent use to select an action?
2. What information does it **ignore** compared to a model-based agent?
3. On what types of grids does your rule perform well?
4. On what types of grids does your rule perform poorly, and why?

*Your answer here.*


---
## 8. Model-Based Agent

### Concept

A **model-based reflex agent** (AIMA §2.4.2) maintains **internal state** that persists across steps. It updates this state using:

- The current percept
- Its model of how actions affect the world

In this lab, the agent builds a partial map by recording every location it has visited. It uses that map to prefer unvisited squares, systematically reducing redundant exploration.

### What to store

```python
self.visited: set          # set of (x, y) positions already seen
self.known_status: dict    # self.known_status[location] = "Clean" or "Dirty"
```

### Required behavior

| Situation | Action |
|---|---|
| Current square is dirty | Return `"Suck"` (and update `known_status`) |
| At least one accessible neighbor is unvisited | Move to one of them |
| All neighbors have been visited (stuck) | **Fallback** — see below |
| No accessible neighbors at all | Return `"NoOp"` |

### Fallback when stuck (TODO 8)

**Minimum (acceptable):** choose any legal neighbor at random.  
**Stronger (recommended):** use a `self.path_stack` to backtrack along the path you came from, enabling depth-first coverage of the whole grid.

The backtracking idea:
- Each time you move to an unvisited square, push your *current* location onto the stack
- When stuck, pop from the stack and move back toward that saved location
- Continue until you find a branch with an unvisited neighbor, or the stack empties

You will need a helper like `action_toward(current, target) -> str` that converts two adjacent positions into a direction string.

> **Note:** The model-based agent still only receives `percept` — it does **not** inspect `environment.status`. The `environment` argument is passed only so you can call `environment.get_neighbors()`.


In [ ]:
class ModelBasedVacuumAgent(Agent):
    """
    Maintains a visited set and (optionally) a path stack to enable
    systematic, backtracking-capable exploration.
    """
    name = "Model-Based Vacuum Agent"

    def __init__(self):
        self.visited: set = set()
        self.known_status: Dict[Position, str] = {}
        self.path_stack: List[Position] = []   # for the backtracking extension

    # ── TODO 7a (optional helper) ──────────────────────────────────────
    # If you implement backtracking, you'll need to convert two adjacent
    # positions into a direction string.  Fill this in if you want to use it.
    #
    # def action_toward(self, current: Position, target: Position) -> str:
    #     cx, cy = current
    #     tx, ty = target
    #     if tx == cx + 1: return "Right"
    #     if tx == cx - 1: return "Left"
    #     if ty == cy + 1: return "Down"
    #     if ty == cy - 1: return "Up"
    #     return "NoOp"

    def choose_action(self, percept: Percept, environment: VacuumEnvironment) -> str:
        location = percept.location

        # Update internal model
        self.visited.add(location)
        self.known_status[location] = percept.status

        # ── TODO 7: Handle dirty square ───────────────────────────────
        # If the current square is dirty:
        #   - update known_status to "Clean"
        #   - return "Suck"
        #
        # ── your code here ──


        # ──────────────────────────────────────────────────────────────

        neighbors = environment.get_neighbors(location)

        # Build list of (action, next_location) pairs for unvisited neighbors
        unvisited = [
            (action, next_loc)
            for action, next_loc in neighbors.items()
            if next_loc not in self.visited
        ]

        # ── TODO 7 (continued): Move to an unvisited neighbor ─────────
        # If unvisited is non-empty:
        #   - (Optional) push current location onto self.path_stack
        #   - return the action toward any one unvisited neighbor
        #     (e.g., unvisited[0][0])
        #
        # ── your code here ──


        # ──────────────────────────────────────────────────────────────

        # ── TODO 8: Fallback when all neighbors are visited ───────────
        #
        # Minimum version:
        #   if neighbors:
        #       return random.choice(list(neighbors.keys()))
        #   return "NoOp"
        #
        # Stronger version (backtracking):
        #   while self.path_stack:
        #       prev = self.path_stack.pop()
        #       if prev in neighbors.values():
        #           return self.action_toward(location, prev)
        #   return "NoOp"
        #
        # ── your code here ──


        # ──────────────────────────────────────────────────────────────

        return "NoOp"   # safety net — you should not reach here often


# Test on a world with obstacles
model_env = VacuumEnvironment(
    width=5, height=5, dirt_probability=0.35, obstacle_probability=0.10, seed=303
)
model_result = run_simulation(ModelBasedVacuumAgent(), model_env, max_steps=100, verbose=False)
print("Model-based agent result:")
print({k: v for k, v in model_result.items() if k not in ("score_history","dirty_history","action_history")})


### ✏️ Checkpoint 3

Answer in **4–6 sentences** using AIMA vocabulary:

1. What internal state does your model-based agent store, and how is it updated?
2. How does that memory change the agent's behavior compared to the reflex agent?
3. Does this agent have a **complete** model of the environment? Why or why not?
4. What fallback did you implement, and what weakness does it still have?

*Your answer here.*


---
## 9. Goal-Based Agent with BFS

### Concept

A **goal-based agent** (AIMA §2.4.3) does not just react or remember — it reasons about a **desired future state** and selects actions that move toward it.

In this lab, the goal is: *"reach and clean the nearest dirty square."*

The agent uses **breadth-first search (BFS)** to find the shortest path to that goal. BFS is appropriate here because the grid is unweighted (every move costs the same 1 point) and we want the nearest dirty square, not just any dirty square.

### Important design note

This agent is permitted to inspect `environment.status` directly — it can see the full grid. This is a simplification that makes sense as a **bridge to Lab 02 (search)**. In a more realistic agent, the goal-based agent would only reason over what it has perceived. Keep this limitation in mind when answering Checkpoint 4.

### BFS algorithm outline

```
frontier = deque([(start_location, [])])    # (location, path_so_far)
visited  = {start_location}

while frontier is not empty:
    location, path_so_far = frontier.popleft()

    if location is a dirty square:
        return path_so_far          # ← first action in this list is what to return

    for each accessible neighbor and the action to reach it:
        if neighbor not in visited:
            visited.add(neighbor)
            frontier.append((neighbor, path_so_far + [action]))

return []   # no reachable dirty square
```

**Why store `path_so_far` instead of a `came_from` dict?**  
Both approaches are correct. Storing the path in the frontier is simpler to read; the `came_from` approach is more memory-efficient for large grids.

### Your task

Complete `bfs_to_nearest_dirty` below. The `choose_action` wrapper is already written for you.


In [ ]:
class GoalBasedVacuumAgent(Agent):
    """
    Plans a path to the nearest dirty square using BFS, then executes
    the first step of that path.  Re-plans at every step.
    """
    name = "Goal-Based Vacuum Agent"

    def choose_action(self, percept: Percept, environment: VacuumEnvironment) -> str:
        # Always clean immediately if standing on a dirty square
        if percept.status == "Dirty":
            return "Suck"

        # Find all dirty squares (agent sees full grid — acknowledged simplification)
        dirty_locations = [
            loc for loc, status in environment.status.items()
            if status == "Dirty"
        ]

        if not dirty_locations:
            return "NoOp"

        path = self.bfs_to_nearest_dirty(percept.location, dirty_locations, environment)

        if path:
            return path[0]   # execute only the first step; re-plan next turn

        return "NoOp"   # no reachable dirty square (blocked by obstacles)

    def bfs_to_nearest_dirty(
        self,
        start: Position,
        dirty_locations: List[Position],
        environment: VacuumEnvironment,
    ) -> List[str]:
        """
        Return the shortest sequence of actions from `start` to the
        nearest location in `dirty_locations`, or [] if none is reachable.

        Use environment.get_neighbors(location) for neighbor expansion —
        it already filters out obstacles and out-of-bounds squares.
        """
        dirty_set = set(dirty_locations)

        # ── TODO 9: Implement BFS ──────────────────────────────────────
        #
        # Initialize:
        #   frontier = deque([(start, [])])   # (current_location, path_so_far)
        #   visited  = {start}                # mark start as visited BEFORE the loop
        #
        # Loop:
        #   pop the leftmost item from the frontier
        #   if the current location is in dirty_set → return path_so_far
        #   for each (action, next_location) in environment.get_neighbors(location).items():
        #       if next_location not in visited:
        #           add next_location to visited
        #           append (next_location, path_so_far + [action]) to frontier
        #
        # ── your code here ──


        # ──────────────────────────────────────────────────────────────

        return []   # no path found


# Test on the same obstacle world as the model-based agent
goal_env = VacuumEnvironment(
    width=5, height=5, dirt_probability=0.35, obstacle_probability=0.10, seed=404
)
goal_result = run_simulation(GoalBasedVacuumAgent(), goal_env, max_steps=100, verbose=False)
print("Goal-based agent result:")
print({k: v for k, v in goal_result.items() if k not in ("score_history","dirty_history","action_history")})


### ✏️ Checkpoint 4

Answer in **4–6 sentences** using AIMA vocabulary:

1. What is the goal state this agent is pursuing?
2. Why is BFS the right search algorithm for this environment? What property of the grid makes it appropriate?
3. What information does the goal-based agent use that the model-based agent did not?
4. Is the performance comparison between goal-based and model-based agents fully fair? Explain the asymmetry.

*Your answer here.*


---
## 10. Controlled Experiments

To compare agents fairly, every agent must face **exactly the same world** in each trial.

`evaluate_agents` creates a world, snapshots it with `copy_world()`, then uses `load_world()` to restore the identical state before each agent runs. This avoids the RNG advancing differently for each agent.

**Run the experiment and inspect the per-agent summary.** You'll use these numbers in Checkpoint 5 and the final reflection.


In [ ]:
def evaluate_agents(
    agent_factories,
    trials: int = 30,
    max_steps: int = 100,
    width: int = 5,
    height: int = 5,
    dirt_probability: float = 0.35,
    obstacle_probability: float = 0.10,
) -> List[dict]:
    """
    Run each agent on identical worlds across multiple trials.
    agent_factories: list of zero-argument callables that return a fresh agent.
    """
    rows = []

    for trial in range(trials):
        # Create one canonical world for this trial
        base_env = VacuumEnvironment(
            width=width, height=height,
            dirt_probability=dirt_probability,
            obstacle_probability=obstacle_probability,
            seed=1000 + trial,
        )
        status, obstacles = base_env.copy_world()

        # Run every agent on the SAME world
        for make_agent in agent_factories:
            env = VacuumEnvironment(width=width, height=height)
            env.load_world(status, obstacles)
            agent = make_agent()
            result = run_simulation(agent, env, max_steps=max_steps)
            result["trial"] = trial
            rows.append(result)

    return rows


def summarize_results(results: List[dict]) -> Dict[str, dict]:
    """Compute per-agent averages across all trials."""
    summary = {}
    for name in sorted({r["agent"] for r in results}):
        rows = [r for r in results if r["agent"] == name]
        summary[name] = {
            "avg_score":      sum(r["score"]      for r in rows) / len(rows),
            "avg_steps":      sum(r["steps"]      for r in rows) / len(rows),
            "avg_cleaned":    sum(r["cleaned"]    for r in rows) / len(rows),
            "avg_dirty_left": sum(r["dirty_left"] for r in rows) / len(rows),
        }
    return summary


agent_factories = [
    lambda: RandomVacuumAgent(),
    lambda: ReflexVacuumAgent(),
    lambda: ModelBasedVacuumAgent(),
    lambda: GoalBasedVacuumAgent(),
]

results = evaluate_agents(agent_factories, trials=30, max_steps=100)
summary = summarize_results(results)

# Display summary table
print(f"{'Agent':<30} {'Avg Score':>10} {'Avg Steps':>10} {'Avg Cleaned':>12} {'Avg Dirty Left':>15}")
print("-" * 80)
for name, stats in summary.items():
    print(
        f"{name:<30} {stats['avg_score']:>10.1f} {stats['avg_steps']:>10.1f} "
        f"{stats['avg_cleaned']:>12.1f} {stats['avg_dirty_left']:>15.1f}"
    )


---
## 11. Plot Your Results

Create at least these two plots. You may add more if they support your analysis in Checkpoint 5.


In [ ]:
def plot_summary_metric(
    summary: Dict[str, dict],
    metric: str,
    title: str,
    ylabel: str,
    higher_is_better: bool = True,
) -> None:
    names  = list(summary.keys())
    values = [summary[name][metric] for name in names]
    colors = ["steelblue"] * len(names)

    # Highlight the best performer
    best_idx = values.index(max(values) if higher_is_better else min(values))
    colors[best_idx] = "darkorange"

    plt.figure(figsize=(9, 4))
    bars = plt.bar(names, values, color=colors)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(rotation=20, ha="right")
    # Annotate bar heights
    for bar, val in zip(bars, values):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f"{val:.1f}",
            ha="center", va="bottom", fontsize=9,
        )
    plt.tight_layout()
    plt.show()


plot_summary_metric(
    summary, "avg_score",
    "Average Score by Agent (30 trials, 5×5 grid, 35% dirt, 10% obstacles)",
    "Average final score",
    higher_is_better=True,
)

plot_summary_metric(
    summary, "avg_dirty_left",
    "Average Dirty Squares Left by Agent",
    "Average dirty squares remaining",
    higher_is_better=False,
)


### ✏️ Checkpoint 5

Answer in **5–7 sentences** using your actual experimental numbers:

1. Which agent achieved the highest average score, and which left the least dirt?
2. Did the highest-scoring agent also use the fewest steps? What does that tell you about the performance measure?
3. How did obstacles affect the different agent types differently?
4. The goal-based agent has access to information the others do not. How does this affect your interpretation of the results?
5. If you changed the scoring so that bumps cost −10 instead of −2, which agent's ranking would change the most, and why?

*Your answer here.*


---
## 12. Extension Challenge (Optional)

Choose one if you finish early. Document your changes and observations in the cell below.

### Option A — Backtracking Model-Based Agent
If you only implemented the basic fallback in Section 8, upgrade it now with a `path_stack` that backtracks along the agent's exploration history. Re-run the experiments and compare the improved model-based agent against the original.

### Option B — Dynamic Environment
Modify `execute_action` so that after each step, each clean non-obstacle square has a 2% chance of becoming dirty again. How does this affect each agent? Which agent design handles it best, and why?

### Option C — New Performance Measure
Redesign the scoring: make movement free (0 cost), make bumps cost −10, and add a +50 bonus for clearing all dirt. Re-run `evaluate_agents` with the new scoring. Does the agent ranking change? Why?

### Option D — Larger Grid
Re-run `evaluate_agents` on a 10×10 grid with the same probabilities. Report how each agent's performance scales. Does the goal-based agent's advantage grow or shrink relative to the model-based agent?


In [ ]:
# Extension workspace
# Label your extension clearly (e.g., "Option A: Backtracking Model-Based Agent")



---
## 13. Final Reflection

Write **one well-developed paragraph** (6–10 sentences) addressing the prompt below.

> In this Vacuum World, what does it mean for an agent to be **rational** (in the AIMA sense)?  
> Is the highest-scoring agent in your experiments always the most rational agent?  
> Use your experimental results as evidence and be precise about the role of the **performance measure**, **percepts**, and **prior knowledge** in your argument.

*Your reflection here.*

---

## Submission Checklist

Before submitting, verify every item:

- [ ] Kernel restarted and notebook runs top-to-bottom without errors (*Kernel → Restart & Run All*)
- [ ] PEAS table (Section 2a) complete — no "TODO" cells remaining
- [ ] Environment property table (Section 2b) complete with justifications
- [ ] `VacuumEnvironment` smoke test shows nonzero dirty and obstacle counts (Section 3)
- [ ] Reflex agent uses a deterministic rule with no stored state (Section 7)
- [ ] Model-based agent stores and uses `self.visited` (Section 8)
- [ ] Goal-based agent BFS returns a valid path (Section 9)
- [ ] Experiment summary table is visible (Section 10)
- [ ] Both plots are visible and labeled (Section 11)
- [ ] All five checkpoints answered with complete sentences and AIMA vocabulary
- [ ] Final reflection paragraph complete (Section 13)
